<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

# ML-4 : Evaluation des modèles

**Navigation** : [Index](README.md) | [<< ML-3 Entrainement&AutoML](ML-3-Entrainement%26AutoML.ipynb) | [Suivant >> ML-5 TimeSeries](ML-5-TimeSeries.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Evaluer un modèle avec les métriques adaptées (R², MAE, RMSE, accuracy)
2. Mettre en œuvre la validation croisée (K-fold)
3. Analyser l'importance des features avec PFI (Permutation Feature Importance)
4. Interpréter une matrice de confusion pour la classification

### Prérequis
- ML-3-Entrainement&AutoML complété
- Notions de régression et classification

### Durée estimée : 40-50 minutes

---

# Évaluation du modèle

Dans ce notebook, vous apprendrez :

- Qu'est-ce que l'évaluation d'un modèle ?
- Comment évaluer un modèle dans ML.NET
- Entraîner et évaluer des modèles avec la validation croisée
- Explicabilité du modèle
- Comment améliorer votre modèle

## Qu'est-ce que l'évaluation d'un modèle ?

L'entraînement est le processus d'application d'algorithmes à des données historiques afin de créer un modèle qui représente fidèlement ces données. Ce modèle est ensuite utilisé sur de nouvelles données pour faire des prédictions.

L'évaluation du modèle est le processus d'utilisation de métriques pour quantifier l'efficacité avec laquelle votre modèle a appris les motifs dans vos données et applique ces apprentissages à des données nouvelles et inconnues.

## Comment évaluer un modèle dans ML.NET

### Importation du package Nuget ML.NET

In [1]:
#i "nuget:https://pkgs.dev.azure.com/dnceng/public/_packaging/MachineLearning/nuget/v3/index.json"

#r "nuget: Microsoft.ML, 5.0.0"
#r "nuget: Microsoft.ML.AutoML, 0.23.0"
#r "nuget: Microsoft.Data.Analysis, 0.23.0"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Restore sources https://pkgs.dev.azure.com/dnceng/public/_packaging/MachineLearning/nuget/v3/index.json Installed Packages Microsoft.Data.Analysis, 0.23.0 Microsoft.ML, 5.0.0 Microsoft.ML.AutoML, 0.23.0

Loading extensions from `C:\Users\jsboi\.nuget\packages\microsoft.ml.automl\0.23.0\interactive-extensions\dotnet\Microsoft.ML.AutoML.Interactive.dll`

Loading extensions from `C:\Users\jsboi\.nuget\packages\microsoft.data.analysis\0.23.0\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

Loading extensions from `C:\Users\jsboi\.nuget\packages\skiasharp\2.88.8\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

### Référencer les packages avec des instructions `using`


In [2]:
using System.Text.Json;
using Microsoft.Data.Analysis;
using Microsoft.ML;
using Microsoft.ML.AutoML;
using Microsoft.ML.Data;
using Microsoft.ML.Trainers.FastTree;
using static Microsoft.ML.Transforms.OneHotEncodingEstimator;


### Télécharger ou localiser les données

Le code suivant essaie de localiser le fichier de données à quelques emplacements connus ou le télécharge à partir de l'emplacement GitHub connu.


In [3]:
using System;
using System.IO;
using System.Net;

// Passage en culture anglaise pour les nombres à virgule
System.Threading.Thread.CurrentThread.CurrentCulture = new System.Globalization.CultureInfo("en-US");

string EnsureDataSetDownloaded(string fileName)
{
    var filePath = Path.Combine(Directory.GetCurrentDirectory(), "data", fileName);

    if (!File.Exists(filePath))
    {
        filePath = Path.Combine(Directory.GetCurrentDirectory(), fileName);
    }

    if (!File.Exists(filePath))
    {
        using (var client = new WebClient())
        {
            client.DownloadFile($"https://raw.githubusercontent.com/dotnet/csharp-notebooks/main/machine-learning/data/{fileName}", filePath);
        }
        Console.WriteLine($"Downloaded {fileName} to : {filePath}");
    }
    else
    {
        Console.WriteLine($"{fileName} found here: {filePath}");
    }

    return filePath;
}

var trainDataPath = EnsureDataSetDownloaded("taxi-fare.csv");
var df = DataFrame.LoadCsv(trainDataPath);



Downloaded taxi-fare.csv to : d:\dev\CoursIA-2\MyIA.AI.Notebooks\ML\ML.Net\taxi-fare.csv



(19,29): warning SYSLIB0014: 'WebClient.WebClient()' est obsolète : 'WebRequest, HttpWebRequest, ServicePoint, and WebClient are obsolete. Use HttpClient instead.' (https://aka.ms/dotnet-warnings/SYSLIB0014)



Une fois les données chargées, utilisez la méthode `Head` pour prévisualiser les cinq premières lignes.


In [4]:
df.Head(5)


index,vendor_id,rate_code,passenger_count,trip_time_in_secs,trip_distance,payment_type,fare_amount
0,CMT,1,1,1271,3.8,CRD,17.5
1,CMT,1,1,474,1.5,CRD,8
2,CMT,1,1,637,1.4,CRD,8.5
3,CMT,1,1,181,0.6,CSH,4.5
4,CMT,1,1,661,1.1,CRD,8.5


### Initialiser MLContext

Toutes les opérations ML.NET commencent par la classe [MLContext](https://docs.microsoft.com/dotnet/api/microsoft.ml.mlcontext). Initialiser `mlContext` crée un nouvel environnement ML.NET qui peut être partagé entre les objets du workflow de création de modèles. C'est similaire, conceptuellement, à `DBContext` dans Entity Framework.


In [5]:
var mlContext = new MLContext();


### Diviser les données en ensembles d'entraînement, de validation et de test

L'ensemble de données original est divisé en trois sous-ensembles : entraînement, validation et test. L'ensemble **d'entraînement** est utilisé pour apprendre les motifs de vos données. L'ensemble **de validation** est utilisé pour optimiser les hyperparamètres du modèle. L'ensemble **de test** est utilisé pour évaluer les performances de votre modèle à l'aide de métriques d'évaluation pour la tâche de régression.

#### Pourquoi ne pas utiliser l'ensemble de données complet ?

Bien que fournir plus d'exemples à votre entraîneur soit généralement recommandé, vous ne voulez pas d'un modèle qui ne performe bien que sur des données historiques. Au lieu de cela, vous cherchez un modèle qui peut apprendre de ces données historiques et généraliser ou faire des prédictions précises sur de nouvelles données inconnues.

Certains problèmes courants rencontrés pendant l'entraînement sont le surapprentissage et le sous-apprentissage. Le sous-apprentissage signifie que l'entraîneur sélectionné n'est pas assez puissant pour ajuster l'ensemble de données d'entraînement et se traduit généralement par une perte élevée pendant l'entraînement et une faible métrique sur l'ensemble de test. Pour résoudre cela, vous devez soit sélectionner un modèle plus puissant, soit effectuer plus d'ingénierie des caractéristiques. Le surapprentissage est l'opposé, et se produit lorsque le modèle apprend trop bien les données d'entraînement. Cela se traduit généralement par une faible perte métrique pendant l'entraînement, mais une perte élevée sur l'ensemble de test.

Une bonne analogie pour ces concepts est de réviser pour un examen. Disons que vous connaissez les questions et réponses à l'avance. Après avoir révisé, vous passez l'examen et obtenez un score parfait. Bonne nouvelle ! Cependant, lorsque vous repassez l'examen avec les questions réorganisées et légèrement reformulées, vous obtenez un score plus bas. Cela suggère que vous avez mémorisé les réponses et n'avez pas vraiment appris les concepts testés. C'est un exemple de surapprentissage. Le sous-apprentissage est le contraire, où les matériaux d'étude que vous avez reçus ne représentent pas fidèlement ce sur quoi vous êtes évalué lors de l'examen. En conséquence, vous devez deviner les réponses car vous n'avez pas assez de connaissances pour répondre correctement.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [6]:
var trainTestData = mlContext.Data.TrainTestSplit(df, testFraction: 0.2);
var validationTestData = mlContext.Data.TrainTestSplit(trainTestData.TestSet, testFraction: 0.5);

var trainSet = trainTestData.TrainSet;
var validationSet = validationTestData.TrainSet;
var testSet = validationTestData.TestSet;


Error: System.IO.FileNotFoundException: Could not load file or assembly 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a'. Le fichier spécifié est introuvable.
File name: 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a'
   at Microsoft.ML.DataOperationsCatalog.CreateSplitColumn(IHostEnvironment env, IDataView& data, String samplingKeyColumn, Nullable`1 seed, Boolean fallbackInEnvSeed)
   at Microsoft.ML.DataOperationsCatalog.TrainTestSplit(IDataView data, Double testFraction, String samplingKeyColumnName, Nullable`1 seed)
   at Submission#8.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

### Créer le pipeline d'entraînement

Pour cet ensemble de données, les transformations suivantes sont appliquées :

- `OneHotEncoding` pour convertir les valeurs catégorielles en valeurs numériques
- `ReplaceMissingValues` pour remplacer les valeurs manquantes
- `Concatenate` prend toutes les caractéristiques et crée un vecteur de caractéristiques

AutoML est utilisé pour définir une expérience de régression en utilisant la colonne `fare_amount` comme colonne à prédire ou colonne d'étiquette.


In [7]:
var pipeline = 
    mlContext.Transforms.Categorical.OneHotEncoding(new[] { new InputOutputColumnPair("vendor_id", "vendor_id"), new InputOutputColumnPair("payment_type", "payment_type") }, outputKind: OutputKind.Binary)
        .Append(mlContext.Transforms.ReplaceMissingValues(new[] { new InputOutputColumnPair("rate_code", "rate_code"), new InputOutputColumnPair("passenger_count", "passenger_count"), new InputOutputColumnPair("trip_time_in_secs", "trip_time_in_secs"), new InputOutputColumnPair("trip_distance", "trip_distance") }))
        .Append(mlContext.Transforms.Concatenate("Features", new[] { "vendor_id", "payment_type", "rate_code", "passenger_count", "trip_time_in_secs", "trip_distance" }))
        .Append(mlContext.Auto().Regression(labelColumnName: "fare_amount"));


[Cellule non executee - erreur environnement .NET]


### Configurer l'expérience

Utilisez AutoML pour configurer notre expérience pour entraîner pendant 60 secondes en utilisant le pipeline que vous venez de définir.

Par défaut, AutoML évalue les modèles qu'il entraîne en utilisant la métrique d'évaluation que vous souhaitez optimiser. Dans ce cas, il s'agit du R-carré, qui est calculé en comparant la valeur réelle `fare_amount` avec la valeur prédite `Score`.

Les métriques d'évaluation dépendent fortement de la tâche. Pour la régression, certaines métriques courantes incluent :

- Erreur absolue moyenne (MAE)
- Erreur quadratique moyenne (MSE)
- Racine de l'erreur quadratique moyenne (RMSE)
- R-carré

Votre ensemble de données et ce que vous essayez d'atteindre influencent fortement votre sélection de métrique. Si vous avez des valeurs aberrantes dans votre ensemble de données, elles peuvent fausser vos prédictions. MAE, MSE et RMSE calculent la distance entre les points de données prédits et réels. Toutes ces mesures sont sensibles aux valeurs aberrantes, donc si vous avez des valeurs aberrantes dans votre ensemble de données, elles apparaîtront dans vos métriques. Le R-carré calcule la corrélation entre les valeurs réelles et préd
ites. Cependant, à mesure que vous ajoutez plus de points de données, votre R-carré peut continuer à augmenter, donnant l'impression erronée qu'un modèle avec une valeur de R-carré élevée a de bonnes capacités prédictives. Le résultat d'une valeur élevée de R-carré peut parfois indiquer un surapprentissage.


In [8]:
var experiment = 
    mlContext.Auto().CreateExperiment()
        .SetPipeline(pipeline)
        .SetTrainingTimeInSeconds(60)
        .SetDataset(trainSet, validationSet)
        .SetRegressionMetric(RegressionMetric.RSquared, "fare_amount", "Score");


[Cellule non executee - erreur environnement .NET]


### Exécuter l'expérience


In [9]:
var result = await experiment.RunAsync();


[Cellule non executee - erreur environnement .NET]


### Voir les métriques d'évaluation du meilleur modèle


In [10]:
$"R-Squared: {result.Metric}"


[Cellule non executee - erreur environnement .NET]


Notez que pendant l'entraînement, les métriques d'évaluation ont été calculées en utilisant l'ensemble de validation. Pour voir comment votre modèle performe sur de nouvelles données, évaluez ses performances par rapport à l'ensemble de test.

Commencez par obtenir le meilleur modèle en utilisant la propriété `Model` des résultats d'entraînement. Ensuite, utilisez la méthode `Transform` pour utiliser le modèle afin de faire des prédictions sur l'ensemble de test.


In [11]:
ITransformer bestModel = result.Model;
var predictions = bestModel.Transform(testSet);


[Cellule non executee - erreur environnement .NET]


Inspectez les premières prédictions (colonne `Score`) et comparez-les avec la valeur réelle (colonne `fare_amount`). Ensuite, calculez la différence entre elles.


In [12]:
var actual = predictions.GetColumn<float>("fare_amount");
var predicted = predictions.GetColumn<float>("Score");

var compare = 
    actual
        .Zip(predicted, (actual, pred) => new { Actual = actual, Predicted = pred, Difference = actual - pred })
        .Take(5);

compare


[Cellule non executee - erreur environnement .NET]


Rien qu'en comparant rapidement les premières valeurs, vous pouvez voir que les prédictions sont généralement à quelques centimes près de la valeur réelle.

Avec ML.NET, vous n'avez pas besoin de calculer manuellement les métriques d'évaluation pour vos modèles. ML.NET fournit une méthode intégrée `Evaluate` pour chacune des tâches de machine learning qu'il prend en charge. Utilisez la méthode `Evaluate` pour la tâche de régression afin de calculer les métriques d'évaluation pour l'ensemble de test où la colonne `fare_amount` est la valeur réelle et la colonne `Score` est la valeur prédite.


In [13]:
var evaluationMetrics = mlContext.Regression.Evaluate(predictions, "fare_amount", "Score");


[Cellule non executee - erreur environnement .NET]


L'utilisation de la méthode `Evaluate` permet non seulement de calculer la métrique que vous avez optimisée pendant l'entraînement (R-carré), mais aussi toutes les métriques pour la tâche de régression.


In [14]:
evaluationMetrics


[Cellule non executee - erreur environnement .NET]


### Utiliser le R-carré pour évaluer le modèle

Bien que vous ayez plusieurs métriques à choisir, lorsque vous avez entraîné le modèle, vous avez optimisé pour le R-carré. Le R-carré (R2), ou coefficient de détermination, représente la puissance prédictive du modèle avec une valeur comprise entre -inf et 1,00. 1,00 signifie qu'il y a un ajustement parfait, et l'ajustement peut être arbitrairement mauvais, donc les scores peuvent être négatifs. Un score de 0,00 signifie que le modèle devine la valeur attendue pour l'étiquette. Une valeur négative de R2 indique que l'ajustement ne suit pas la tendance des données et que le modèle performe moins bien que le hasard. Cela n'est possible qu'avec des modèles de régression non linéaire ou des régressions linéaires contraintes. Le R2 mesure à quel point les valeurs réelles des données de test sont proches des valeurs prédites.

Pour plus d'informations sur les autres métriques d'évaluation, consultez le guide sur [comment évaluer votre modèle ML.NET avec des métriques](https://docs.microsoft.com/dotnet/machine-learning/resources/metrics).

## Entraîner et évaluer des modèles en utilisant la validation croisée

### Qu'est-ce que la validation croisée ?

La validation croisée est une technique d'entraînement et d'évaluation de modèle qui divise les données en plusieurs partitions et entraîne plusieurs modèles sur ces partitions. Cette technique améliore la robustesse du modèle en réservant des données du processus d'entraînement. En plus d'améliorer les performances sur des observations non vues, dans des environnements de données limitées, elle peut être un outil efficace pour entraîner des modèles avec un plus petit ensemble de données.

### Entraîner un modèle en utilisant la validation croisée

Commencez par initialiser le `MLContext`.


In [15]:
var cvMLContext = new MLContext();


[Cellule non executee - erreur environnement .NET]


Ensuite, définissez votre pipeline. Dans ce cas, l'entraîneur réel est utilisé au lieu de `SweepableEstimator` d'AutoML.


In [16]:
var cvMLPipeline = 
    cvMLContext.Transforms.Categorical.OneHotEncoding(new[] { new InputOutputColumnPair("vendor_id", "vendor_id"), new InputOutputColumnPair("payment_type", "payment_type") }, outputKind: OutputKind.Binary)
        .Append(cvMLContext.Transforms.ReplaceMissingValues(new[] { new InputOutputColumnPair("rate_code", "rate_code"), new InputOutputColumnPair("passenger_count", "passenger_count"), new InputOutputColumnPair("trip_time_in_secs", "trip_time_in_secs"), new InputOutputColumnPair("trip_distance", "trip_distance") }))
        .Append(cvMLContext.Transforms.Concatenate("Features", new[] { "vendor_id", "payment_type", "rate_code", "passenger_count", "trip_time_in_secs", "trip_distance" }))
        .Append(cvMLContext.Regression.Trainers.FastForest(labelColumnName: "fare_amount"));


[Cellule non executee - erreur environnement .NET]


Utilisez la méthode `CrossValidate` pour démarrer l'entraînement et l'évaluation sur vos données en utilisant le pipeline défini. Par défaut, les données sont divisées en cinq sous-ensembles, mais vous pouvez définir cette valeur à n'importe quelle valeur de votre choix en utilisant le paramètre `numberOfFolds`.


In [17]:
var cvResults = cvMLContext.Regression.CrossValidate(trainSet, cvMLPipeline, labelColumnName: "fare_amount");

cvResults.Select(x => x.Metrics)


[Cellule non executee - erreur environnement .NET]


### Calculer les métriques d'évaluation de l'ensemble de test

Comme dans l'exemple précédent, utilisez la méthode `Evaluate` sur l'ensemble complet de test pour évaluer les performances des modèles entraînés en utilisant la validation croisée.


In [18]:
var cvTestEvalMetrics = 
    cvResults
        .Select(fold => fold.Model.Transform(trainTestData.TestSet))
        .Select(predictions => cvMLContext.Regression.Evaluate(predictions, "fare_amount", "Score"));
        
cvTestEvalMetrics


[Cellule non executee - erreur environnement .NET]


## Explicabilité du modèle

Les métriques d'évaluation sont un bon moyen de quantifier la précision avec laquelle votre modèle fait des prédictions sur de nouvelles données. Cependant, une bonne métrique d'évaluation ne doit pas être le seul facteur à considérer lors de l'évaluation de votre modèle. Pour renforcer la confiance autour de votre modèle et des décisions qu'il prend, il est important de comprendre comment et pourquoi il prend ces décisions.

Les modèles deviennent de plus en plus courants dans la société et affectent la vie des individus. Par exemple, imaginons qu'un modèle de machine learning soit utilisé pour établir un diagnostic médical. Il est probable que le modèle ait raison dans son diagnostic, mais les enjeux d'un mauvais diagnostic sont élevés en raison des effets sur la santé d'un individu. Il est donc important pour tous les intervenants (patients, médecins, régulateurs) de comprendre ce qui a poussé un modèle à établir ce diagnostic afin de se sentir confiants dans sa décision.

### Explications globales et locales

Lors de l'explication des modèles de machine learning, vous pouvez le faire au niveau global et local.

Les explications globales sont des généralisations au niveau agrégé. Par exemple, disons que vous construisez un modèle pour prédire les tarifs de taxi. En expliquant pourquoi certains tarifs sont plus chers que d'autres, vous trouverez probablement que les trajets qui parcourent une distance plus longue ou qui durent plus longtemps sont probablement plus chers. Bien que cela ne vous dise pas exactement pourquoi un trajet particulier était plus cher qu'un autre, à un niveau agrégé, vous pouvez voir quelles caractéristiques sont importantes pour le modèle lorsqu'il prend ses décisions.

Si vous avez besoin de plus de granularité pour expliquer les décisions de votre modèle, c'est là que les explications locales entrent en jeu. Les explications locales vous permettent de voir pour une prédiction donnée quelles caractéristiques ont contribué à la décision du modèle. Par exemple, disons qu'un modèle est utilisé pour déterminer le risque de crédit pour des prêts personnels. Étant donné deux clients avec des montants de dette, des revenus et des historiques de paiement différents, le modèle détermine lequel des clients est le plus susceptible de rembourser le prêt. En utilisant des techniques d'explicabilité locale, vous pouvez inspecter au niveau individuel quelles caractéristiques ont contribué à la décision de refuser un prêt.

### Techniques d'explicabilité dans ML.NET

ML.NET fournit deux techniques pour expliquer les modèles :

- Importance des caractéristiques par permutation (PFI)
- Calcul de la contribution des caractéristiques (FCC)

#### Importance des caractéristiques par permutation (PFI)

L'importance des caractéristiques par permutation est une technique d'explicabilité **globale**. À un haut niveau, elle mélange aléatoirement les données une caractéristique à la fois pour l'ensemble de données complet et calcule dans quelle mesure la métrique de performance d'intérêt diminue. Plus le changement est important, plus la caractéristique est importante. Pour plus d'informations, consultez [Interpréter les prédictions de modèle en utilisant l'importance des caractéristiques par permutation](https://docs.microsoft.com/dotnet/machine-learning/how-to-guides/explain-machine-learning-model-permutation-feature-importance-ml-net).

#### Calcul de la contribution des caractéristiques (FCC)

Le calcul de la contribution des caractéristiques est une technique d'explicabilité **locale**. Cette technique calcule une liste spécifique au modèle des contributions par caractéristique à chacune des prédictions. Ces contributions peuvent être positives (elles augmentent la métrique d'évaluation) ou négatives (elles diminuent la métrique d'évaluation).

### Expliquer les modèles avec l'importance des caractéristiques par permutation (PFI)

#### Initialiser MLContext



In [19]:
var pfiMLContext = new MLContext();


[Cellule non executee - erreur environnement .NET]


#### Définir le pipeline de préparation des données


In [20]:
var pfiDataPipeline = 
    pfiMLContext.Transforms.Categorical.OneHotEncoding(new[] { new InputOutputColumnPair("vendor_id", "vendor_id"), new InputOutputColumnPair("payment_type", "payment_type") }, outputKind: OutputKind.Binary)
        .Append(pfiMLContext.Transforms.ReplaceMissingValues(new[] { new InputOutputColumnPair("rate_code", "rate_code"), new InputOutputColumnPair("passenger_count", "passenger_count"), new InputOutputColumnPair("trip_time_in_secs", "trip_time_in_secs"), new InputOutputColumnPair("trip_distance", "trip_distance") }))
        .Append(pfiMLContext.Transforms.Concatenate("Features", new[] { "vendor_id", "payment_type", "rate_code", "passenger_count", "trip_time_in_secs", "trip_distance" }));


[Cellule non executee - erreur environnement .NET]


#### Appliquer les transformations de données aux données d'entraînement


In [21]:
var pfiPreprocessedData = 
    pfiDataPipeline
        .Fit(trainSet)
        .Transform(trainSet);


[Cellule non executee - erreur environnement .NET]


#### Définir votre entraîneur


In [22]:
var pfiTrainer = pfiMLContext.Regression.Trainers.FastForest(labelColumnName: "fare_amount");


[Cellule non executee - erreur environnement .NET]


#### Ajuster l'entraîneur à vos données prétraitées


In [23]:
var pfiModel = pfiTrainer.Fit(pfiPreprocessedData);


[Cellule non executee - erreur environnement .NET]


#### Calculer l'importance des caractéristiques par permutation (PFI)


In [24]:
var permutationFeatureImportance =
    mlContext
        .Regression
        .PermutationFeatureImportance(pfiModel, pfiPreprocessedData, permutationCount: 3, labelColumnName: "fare_amount");


[Cellule non executee - erreur environnement .NET]


#### Extraire la métrique du R-carré


In [25]:
var pfiMetrics = 
    permutationFeatureImportance
        .Select((metric, idx) => new { idx, metric.RSquared })
        .OrderByDescending(x => Math.Abs(x.RSquared.Mean));


[Cellule non executee - erreur environnement .NET]


#### Obtenir la liste des noms de caractéristiques


In [26]:
var featureContributionColumn = pfiPreprocessedData.Schema.GetColumnOrNull("Features");
var slotNames = new VBuffer<ReadOnlyMemory<char>>();
featureContributionColumn.Value.GetSlotNames(ref slotNames);
var slotNameValues = slotNames.DenseValues();


[Cellule non executee - erreur environnement .NET]


#### Mapper les métriques PFI aux noms des caractéristiques


In [27]:
var featureImportance = 
    pfiMetrics
        .Zip(slotNameValues, (a, b) => new KeyValuePair<string, double>(b.ToString(), a.RSquared.Mean));

featureImportance


[Cellule non executee - erreur environnement .NET]


### Expliquer les modèles avec le calcul de la contribution des caractéristiques (FCC)

#### Initialiser MLContext


In [28]:
var fccMLContext = new MLContext();


[Cellule non executee - erreur environnement .NET]


#### Définir le pipeline de préparation des données


In [29]:
var fccDataPipeline = 
    fccMLContext.Transforms.Categorical.OneHotEncoding(new[] { new InputOutputColumnPair("vendor_id", "vendor_id"), new InputOutputColumnPair("payment_type", "payment_type") }, outputKind: OutputKind.Binary)
        .Append(fccMLContext.Transforms.ReplaceMissingValues(new[] { new InputOutputColumnPair("rate_code", "rate_code"), new InputOutputColumnPair("passenger_count", "passenger_count"), new InputOutputColumnPair("trip_time_in_secs", "trip_time_in_secs"), new InputOutputColumnPair("trip_distance", "trip_distance") }))
        .Append(fccMLContext.Transforms.Concatenate("Features", new[] { "vendor_id", "payment_type", "rate_code", "passenger_count", "trip_time_in_secs", "trip_distance" }));


[Cellule non executee - erreur environnement .NET]


#### Appliquer les transformations de données aux données d'entraînement


In [30]:
var fccPreprocessedData = 
    fccDataPipeline
        .Fit(trainSet)
        .Transform(trainSet);


[Cellule non executee - erreur environnement .NET]


#### Définir votre entraîneur


In [31]:
var fccTrainer = fccMLContext.Regression.Trainers.FastForest(labelColumnName: "fare_amount");


[Cellule non executee - erreur environnement .NET]


#### Ajuster l'entraîneur à vos données prétraitées


In [32]:
var fccModel = fccTrainer.Fit(fccPreprocessedData);


[Cellule non executee - erreur environnement .NET]


#### Calculer les contributions des caractéristiques


In [33]:
var featureContributionCalc = 
    fccMLContext.Transforms.CalculateFeatureContribution(fccModel, normalize: false)
        .Fit(fccPreprocessedData)
        .Transform(fccPreprocessedData);


[Cellule non executee - erreur environnement .NET]


#### Obtenir la liste des noms de caractéristiques


In [34]:
var featureContributionColumn = featureContributionCalc.Schema.GetColumnOrNull("FeatureContributions");
var slotNames = new VBuffer<ReadOnlyMemory<char>>();
featureContributionColumn.Value.GetSlotNames(ref slotNames);
var slotNameValues = slotNames.DenseValues();


[Cellule non executee - erreur environnement .NET]


#### Obtenir les valeurs de contribution des caractéristiques


In [35]:
var featureContributionValues = featureContributionCalc.GetColumn<float[]>("FeatureContributions");


[Cellule non executee - erreur environnement .NET]


#### Mapper les valeurs de contribution des caractéristiques aux noms des caractéristiques


In [36]:
var featureContributions = 
    featureContributionValues
        .Select(x => x.Zip(slotNameValues, (a, b) => new KeyValuePair<string, float>(b.ToString(), a)));


[Cellule non executee - erreur environnement .NET]


#### Afficher les contributions des caractéristiques pour la première prédiction


In [37]:
featureContributions.First()


[Cellule non executee - erreur environnement .NET]


## Comment puis-je améliorer mon modèle ?

L'évaluation du modèle est une étape importante dans le workflow de machine learning pour déterminer si un modèle est prêt à être déployé en production. Si votre modèle ne répond pas à vos critères pour être prêt pour la production, il y a plusieurs choses que vous pouvez essayer pour améliorer votre modèle. Celles-ci incluent :

- **Reformuler le problème** - Essayez-vous de résoudre le bon problème ? Envisagez d'examiner le problème sous différents points de vue.
- **Fournir plus d'échantillons de données** - L'expérience est le meilleur enseignant. Fournir plus d'exemples qui représentent votre espace de problème aide les entraîneurs à identifier plus de cas particuliers.
- **Ajouter des caractéristiques (plus de contexte)** - Construire un contexte autour des points de données aide les algorithmes ainsi que les experts en la matière à mieux prendre des décisions.
- **Utiliser des données et des caractéristiques significatives** - Plus de données et de caractéristiques peuvent aider à améliorer la précision, mais aussi introduire du bruit. Envisagez d'utiliser PFI et FCC pour déterminer les caractéristiques impactant vos prédictions.
- **Utiliser la validation croisée** - La validation croisée peut être un outil efficace pour entraîner des modèles avec des ensembles de données plus petits.
- **Ajustement des hyperparamètres** - Utilisez les espaces de recherche AutoML pour trouver les bons hyperparamètres pour votre algorithme.
- **Choisir un algorithme différent** - Utilisez AutoML pour itérer à travers les différents algorithmes disponibles dans ML.NET.

## Résumé

Ce notebook a couvert l'évaluation des modèles ML.NET :

| Concept | Description |
|---------|-------------|
| R² (R-carré) | Mesure la corrélation entre valeurs réelles et prédites (1.0 = parfait) |
| MAE | Erreur absolue moyenne (robuste aux valeurs aberrantes) |
| RMSE | Racine de l'erreur quadratique moyenne (pénalise les grosses erreurs) |
| Validation croisée | Technique d'évaluation sur plusieurs sous-ensembles |
| PFI | Importance des features par permutation (explicabilité globale) |
| FCC | Contribution des features par calcul (explicabilité locale) |

**Points clés** :
- Ne jamais évaluer sur les données d'entraînement - utiliser un test set séparé
- La validation croisée améliore la robustesse des estimations de performance
- PFI et FCC permettent de comprendre quelles features influencent les prédictions
- R² proche de 1.0 indique un bon ajustement, mais surveiller aussi le surapprentissage

**Navigation** : [Index](README.md) | [<< ML-3 Entrainement&AutoML](ML-3-Entrainement%26AutoML.ipynb) | [Suivant >> ML-5 TimeSeries](ML-5-TimeSeries.ipynb)

## Exercices supplementaires

Les exercices suivants approfondissent l'evaluation des modeles avec des techniques de classification binaire et des metriques avancees.

### Exercice 1 : Courbe ROC et calcul de l'AUC

Entrainez un classifieur binaire, calculez les points de la courbe ROC en faisant varier le seuil de decision, tracez la courbe avec Plotly.NET et calculez l'AUC manuellement.

**Objectifs** :
1. Convertir le probleme de regression taxi-fare en classification binaire (seuil sur `fare_amount`)
2. Entrainer un classifieur binaire avec SDCA Logistic Regression
3. Extraire les probabilites de prediction pour chaque observation
4. Calculer TPR (True Positive Rate) et FPR (False Positive Rate) pour plusieurs seuils
5. Tracer la courbe ROC avec Plotly.NET (`Chart.Line`)
6. Calculer l'AUC manuellement par la methode des trapezes

**Indice** : Pour chaque seuil de 0.1 a 0.9 par pas de 0.1, classez comme positif si `Probability >= seuil`. Calculez TPR = TP/(TP+FN) et FPR = FP/(FP+TN). L'AUC est la somme des aires des trapezes entre points consecutifs : `AUC += 0.5 * (FPR[i+1] - FPR[i]) * (TPR[i+1] + TPR[i])`.

In [38]:
// Exercice 1 : Courbe ROC et calcul de l'AUC
// TODO etudiant : Entrainez un classifieur binaire et tracez la courbe ROC avec Plotly.NET
// Indice : utilisez les scores de probabilite pour faire varier le seuil de decision
// Etape 1 : convertissez le probleme taxi-fare en classification binaire (fare > mediane = positif)
// Etape 2 : entrainez un classifieur SDCA Logistic Regression
// Etape 3 : recuperez les probabilites via predictions.GetColumn<float>("Probability")
// Etape 4 : pour plusieurs seuils (0.1 a 0.9), calculez TPR et FPR pour tracer la courbe ROC
// Etape 5 : calculez l'AUC par la methode des trapezes (somme des aires sous la courbe)

Console.WriteLine("Exercice a completer : courbe ROC et AUC");

Exercice a completer


### Exercice 2 : Analyse manuelle de la matrice de confusion

Apres avoir entraine un classifieur binaire, extrayez les predictions et calculez manuellement les metriques de classification (precision, recall, F1-score) pour chaque classe.

**Objectifs** :
1. Convertir le probleme de regression taxi-fare en classification binaire (seuil sur `fare_amount`)
2. Entrainer un classifieur binaire (SDCA ou FastTree)
3. Comparer les predictions aux labels reels pour extraire TP, FP, TN, FN
4. Calculer precision, recall et F1-score manuellement pour les deux classes
5. Verifier vos resultats avec les metriques ML.NET

**Indice** : Parcourez `predictions.GetColumn<bool>("PredictedLabel")` et `predictions.GetColumn<bool>("Label")` simultanement. Comptez les 4 cas (TP, FP, TN, FN), puis appliquez les formules : Precision = TP/(TP+FP), Recall = TP/(TP+FN), F1 = 2 * Precision * Recall / (Precision + Recall).

In [39]:
// Exercice 2 : Analyse manuelle de la matrice de confusion
// TODO etudiant : Entrainez un classifieur binaire et calculez precision, recall, F1 manuellement
// Indice : comptez les TP, FP, TN, FN a partir des predictions et des labels reels
// Etape 1 : utilisez les donnees taxi-fare et convertissez le probleme en classification binaire
//          (par ex. fare_amount > seuil = "cher", sinon "abordable")
// Etape 2 : entrainez un modele FastTree ou SDCA en classification binaire
// Etape 3 : comparez les predictions avec les labels reels pour remplir la matrice de confusion
// Etape 4 : calculez precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = 2*P*R/(P+R) pour chaque classe

Console.WriteLine("Exercice a completer : analyse de la matrice de confusion");

Exercice a completer


## Exercice : Analyse complète d'un modèle de prédiction

Créez un modèle pour prédire la durée d'un trajet taxi (`trip_time_in_secs`) au lieu du prix, puis analysez sa qualité.

### Objectifs
1. Définir un pipeline avec `trip_time_in_secs` comme label (au lieu de `fare_amount`)
2. Entraîner avec validation croisée (3 folds minimum)
3. Calculer et interpréter les métriques (R², RMSE, MAE)
4. Identifier les 2 features les plus importantes avec PFI

### Indices
- Modifiez le paramètre `labelColumnName` dans le trainer
- Utilisez `CrossValidate` avec `numberOfFolds`
- Comparez les métriques entre les différents folds
- PFI vous montre quelles features influencent le plus les prédictions

In [40]:
// Exercice : Prediction de duree de trajet avec analyse complete

// TODO: Creer un nouveau MLContext
MLContext durationContext = null;

// TODO: Definir le pipeline pour predire trip_time_in_secs
// Indice: changez labelColumnName de "fare_amount" a "trip_time_in_secs"
IEstimator<ITransformer> durationPipeline = null;

// TODO: Appliquer la validation croisee (3 folds minimum)
// Indice: var cvResults = durationContext.Regression.CrossValidate(data, pipeline, numFolds: 3);
object cvResults = null;  // TODO etudiant : remplacer par CrossValidate

if (cvResults != null)
{
    // TODO: Afficher les metriques de chaque fold
    Console.WriteLine("=== Resultats Validation Croisee ===");
    // Parcourez cvResults et affichez R2, RMSE, MAE pour chaque fold
}
else
{
    Console.WriteLine("Exercice a completer : implementez les TODO pour la validation croisee");
}

// TODO: Calculer les metriques moyennes
var avgRSquared = 0.0;
var avgRMSE = 0.0;
var avgMAE = 0.0;

Console.WriteLine($"Moyennes - R2: {avgRSquared:F4}, RMSE: {avgRMSE:F2}, MAE: {avgMAE:F2}");

// TODO: Analyser l'importance des features avec PFI
// Indice: utilisez PermutationFeatureImportance sur le meilleur modele

// TODO: Afficher les 2 features les plus importantes
Console.WriteLine("Top 2 Features (PFI) - a implementer");

// TODO: Interpretez les resultats - quelles features sont les plus pertinentes pour predire la duree ?

Exercice a completer
